# Week 05 - Day 1: Jupyter & NumPy Foundations

## Overview

This notebook documents the practical work for Day 1 of Week 05.

The objective is to develop a reliable and reproducible workflow for
Jupyter-based numerical computing while building foundational NumPy skills
relevant to data processing and AI/ML workflows.

## Learning Objectives

By the end of this notebook, I will be able to:

1. Explain how Jupyter cells and kernel state affect execution.
2. Identify and reproduce a hidden-state execution issue.
3. Create and inspect NumPy arrays using different construction methods.
4. Perform indexing, slicing, and Boolean masking on NumPy arrays.
5. Apply broadcasting to a practical mean-centering operation.
6. Perform axis-based aggregation and interpret the resulting shapes.
7. Investigate reshape behavior and view-versus-copy semantics.
8. Use Jupyter/IPython tools for documentation lookup and performance measurement.
9. Validate the notebook from a clean kernel state using **Restart Kernel → Run All**.

## Environment Verification

### Why this check matters

Before running numerical experiments, I verify the Python and NumPy
environment being used by the notebook.

This helps ensure that the notebook is executing inside the intended
virtual environment rather than relying on a global Python installation.

In [1]:
import sys
import numpy as np

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("NumPy version:", np.__version__)

Python executable: /home/as/Documents/week5/.venv/bin/python
Python version: 3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
NumPy version: 2.2.6


### Interpretation

The notebook is running with the Python interpreter from the project's
virtual environment:

`/home/as/Documents/week5/.venv/bin/python`

This confirms that the notebook is isolated from the system-level Python
environment. NumPy is also successfully available in the same environment,
so the required numerical experiments can be performed reproducibly.

## Hidden Kernel State
I will intentionally create a dependency between two cells and execute
them out of order. I will then restart the kernel and run all cells from
top to bottom to determine whether the notebook is actually reproducible.

In [2]:
#print(message)

In [3]:
message = "Hello from the Jupyter kernel state"

### Observation

The notebook initially appeared to work because the variable `message`
had already been created in the kernel by executing the later cell first.
After restarting the kernel, the previously accumulated state was removed.
Running the notebook from top to bottom exposed the dependency error because
the first cell attempted to use `message` before it was defined.
This demonstrates why a notebook should not depend on hidden or previously
accumulated kernel state.

In [4]:
print(message)

Hello from the Jupyter kernel state


## NumPy Array Creation and Inspection

### Objective

The objective of this section is to create NumPy arrays using different
construction methods and inspect their structural properties.

I will compare:

- `np.array()`
- `np.zeros()`
- `np.ones()`
- `np.arange()`
- `np.linspace()`

For each array, I will inspect its `shape`, `dtype`, and `size`.

### Why these properties matter

- `shape` describes the dimensional structure of the array.
- `dtype` identifies the data type used to store its elements.
- `size` gives the total number of elements.

These properties are important when working with numerical data because
many NumPy operations depend on compatible shapes and data types.

1. `np.array()`
The array contains five integer elements, so its size is `5` and its
one-dimensional shape is `(5,)`.
The dtype reflects how NumPy represents the stored numerical values.

In [5]:
import numpy as np

array_from_list = np.array([10, 20, 30, 40, 50])

print("Array:", array_from_list)
print("Shape:", array_from_list.shape)
print("Dtype:", array_from_list.dtype)
print("Size:", array_from_list.size)

Array: [10 20 30 40 50]
Shape: (5,)
Dtype: int64
Size: 5


2. `np.zeros()` creates an array initialized with zeros. It is useful when
a predefined array structure is required before values are populated.

In [6]:
zeros_array = np.zeros((3, 4))

print("Array:\n", zeros_array)
print("Shape:", zeros_array.shape)
print("Dtype:", zeros_array.dtype)
print("Size:", zeros_array.size)

Array:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Shape: (3, 4)
Dtype: float64
Size: 12


3. `np.ones()` creates an array whose elements are initialized to one.
Like `np.zeros()`, it is useful when a specific array shape is required
before performing further numerical operations.

In [7]:
ones_array = np.ones((2, 3))

print("Array:\n", ones_array)
print("Shape:", ones_array.shape)
print("Dtype:", ones_array.dtype)
print("Size:", ones_array.size)

Array:
 [[1. 1. 1.]
 [1. 1. 1.]]
Shape: (2, 3)
Dtype: float64
Size: 6


4. `np.arange()` generates evenly spaced values based on a start value,
stop value, and step size.

The stop value is excluded from the generated sequence.

In [8]:
range_array = np.arange(0, 10, 2)

print("Array:", range_array)
print("Shape:", range_array.shape)
print("Dtype:", range_array.dtype)
print("Size:", range_array.size)

Array: [0 2 4 6 8]
Shape: (5,)
Dtype: int64
Size: 5


5. `np.linspace()` generates a specified number of evenly spaced values
between a start and an end point.
Unlike `np.arange()`, the third argument represents the number of values,
not the step size.

In [9]:
linear_array = np.linspace(0, 1, 5)

print("Array:", linear_array)
print("Shape:", linear_array.shape)
print("Dtype:", linear_array.dtype)
print("Size:", linear_array.size)

Array: [0.   0.25 0.5  0.75 1.  ]
Shape: (5,)
Dtype: float64
Size: 5


### Key Observation
The array construction method affects the resulting values, dtype, shape,
and size. Therefore, inspecting these properties is important before using
an array in subsequent numerical operations.

## Python List vs NumPy Array Performance
The objective of this experiment is to compare the performance of a
Python list and a NumPy array when performing numerical operations on
1,000,000 values.
### Method
I will:
1. Create 1,000,000 values using a Python list.
2. Create the equivalent NumPy array.
3. Benchmark an element-wise multiplication operation using `%timeit`.
4. Compare the measured execution times.
5. Interpret the result rather than relying only on a general assumption
   that NumPy is faster.

In [10]:
numbers_list = list(range(1_000_000))
numbers_array = np.array(numbers_list)

print("List length:", len(numbers_list))
print("NumPy array shape:", numbers_array.shape)
print("NumPy array size:", numbers_array.size)

List length: 1000000
NumPy array shape: (1000000,)
NumPy array size: 1000000


### Python List Operation

Python lists do not directly support element-wise multiplication by a
scalar. Therefore, a list comprehension is used to perform the same
logical operation on every element.
The benchmark measures the complete list-based computation.

In [11]:
%timeit [x * 2 for x in numbers_list]

82.5 ms ± 1.69 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### NumPy Vectorized Operation
NumPy supports element-wise arithmetic directly on arrays. The
multiplication is therefore expressed as a vectorized operation without
an explicit Python loop.
I will benchmark the same multiplication operation used for the Python
list.

In [12]:
%timeit numbers_array * 2

1.09 ms ± 86.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


###  Results
The measured execution times were:

| Approach | Mean Execution Time |
|---|---:|
| Python list comprehension | 71.5 ms |
| NumPy vectorized operation | 844 μs (0.844 ms) |

### Interpretation

The NumPy vectorized operation completed substantially faster than the
equivalent Python list comprehension in this result.

The Python list implementation required approximately 71.5 ms, whereas
the NumPy operation required approximately 844 μs (0.844 ms).

This demonstrates the performance advantage of NumPy for large-scale
element-wise numerical operations in this environment.

The exact performance difference is hardware- and environment-dependent,
so the measured values should be treated as benchmark observations rather
than universal performance guarantees.

## 2D Indexing and Slicing
The objective of this section is to practice accessing elements,
rows, columns, and sub-regions of a two-dimensional NumPy array.

Indexing and slicing are fundamental operations in data preprocessing.
They allow specific observations, features, rows, columns, or subsets of
a dataset to be selected without modifying the complete array.
I will verify not only the selected values but also the resulting shapes.

The array contains 4 rows and 5 columns, giving a total of 20 elements.
The indexing format for a 2D NumPy array is:
`array[row, column]`
Both row and column indices are zero-based.

In [13]:
matrix = np.arange(1, 21).reshape(4, 5)

print(matrix)
print("Shape:", matrix.shape)

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]]
Shape: (4, 5)


### Accessing a Specific Element
A specific element can be accessed using its row and column indices.
The target in this example is the value `13`.

In [14]:
element = matrix[2, 2]

print("Selected element:", element)
print("Shape:", np.shape(element))

Selected element: 13
Shape: ()


### Selecting a Complete Row
A complete row can be selected by specifying its row index and using
`:` for all columns.

In [15]:
row = matrix[1, :]

print("Selected row:", row)
print("Shape:", row.shape)

Selected row: [ 6  7  8  9 10]
Shape: (5,)


The second row is selected using `matrix[1, :]`.
The resulting array has shape `(5,)` because the selected row contains
five elements.

### Selecting a Complete Column
A complete column can be selected by using `:` for all rows and specifying
the required column index.

In [16]:
column = matrix[:, 2]

print("Selected column:", column)
print("Shape:", column.shape)

Selected column: [ 3  8 13 18]
Shape: (4,)


The third column is selected using `matrix[:, 2]`.
The resulting shape is `(4,)` because the column contains four elements,
one from each row.

### Selecting a Sub-block
Slicing can be used to extract a rectangular region from a 2D array.
The following operation selects rows `1` through `2` and columns `2`
through `4`. The stop indices are exclusive.

In [17]:
sub_block = matrix[1:3, 2:5]

print("Sub-block:")
print(sub_block)
print("Shape:", sub_block.shape)

Sub-block:
[[ 8  9 10]
 [13 14 15]]
Shape: (2, 3)


The slice `matrix[1:3, 2:5]` extracts a rectangular sub-block containing
2 rows and 3 columns.
The resulting shape `(2, 3)` confirms that the selection contains
6 elements.

### Shape Summary

| Operation | Result | Shape |
|---|---|---|
| `matrix[2, 2]` | Single element | `()` |
| `matrix[1, :]` | Complete row | `(5,)` |
| `matrix[:, 2]` | Complete column | `(4,)` |
| `matrix[1:3, 2:5]` | Sub-block | `(2, 3)` |

### Key Takeaway
Indexing can reduce an array to a scalar or a lower-dimensional array,
while slicing can preserve a structured subset of the original data.
Checking `.shape` provides a reliable way to verify what was actually
selected.

## Boolean Masking
The objective of this section is to filter NumPy arrays using Boolean
conditions.
Boolean masking allows values to be selected based on whether they satisfy
a given condition, without explicitly writing a Python loop.
I will first apply a single condition and then combine multiple conditions
using logical operators.

### Filtering Values Using a Single Condition
A Boolean condition applied to a NumPy array produces a Boolean mask.
The mask contains `True` for elements that satisfy the condition and
`False` for elements that do not.

In [18]:
values = np.array([10, 25, 30, 45, 50, 65, 80])

mask = values > 40

print("Boolean mask:", mask)
print("Filtered values:", values[mask])

Boolean mask: [False False False  True  True  True  True]
Filtered values: [45 50 65 80]


The expression `values > 40` creates a Boolean mask. Applying this mask
back to the original array returns only the elements whose corresponding
mask value is `True`.
This provides a concise way to perform condition-based filtering on
numerical data.

### Filtering with Multiple Conditions
Multiple Boolean conditions can be combined to create more specific
filters.
The `&` operator represents an element-wise AND condition.

In [19]:
filtered = values[(values > 30) & (values < 70)]

print("Filtered values:", filtered)

Filtered values: [45 50 65]


In [20]:
filtered_or = values[(values < 20) | (values > 60)]

print("Filtered values:", filtered_or)

Filtered values: [10 65 80]


In [21]:
print("Original values:", values)
print("Number of values > 40:", np.sum(values > 40))
print("Filtered values:", values[values > 40])

Original values: [10 25 30 45 50 65 80]
Number of values > 40: 4
Filtered values: [45 50 65 80]


### Key Takeaway

Boolean masking separates condition creation from data selection:
1. A Boolean expression creates a mask.
2. The mask identifies which elements satisfy the condition.
3. Applying the mask returns the corresponding values.
For multiple conditions, NumPy uses element-wise operators such as `&`
and `|`, with each condition enclosed in parentheses.

## Broadcasting and Mean-Centering
The objective of this section is to understand NumPy broadcasting through
a practical mean-centering operation.
Broadcasting allows NumPy to perform arithmetic operations between arrays
with compatible shapes without explicitly repeating or looping over the
data.
I will:
1. Create a 2D numerical dataset.
2. Calculate the mean of each column.
3. Subtract the column means from the original dataset using broadcasting.
4. Verify the shapes involved.
5. Calculate the means again to confirm that the centered data has
   approximately zero mean in each column.
Mean-centering is a common preprocessing operation in numerical and
machine-learning workflows. Broadcasting allows the same column-level
operation to be applied efficiently across all rows.

In [22]:
data = np.array([
    [10, 20, 30],
    [20, 30, 40],
    [30, 40, 50],
    [40, 50, 60]
])

print("Data:")
print(data)
print("Shape:", data.shape)

Data:
[[10 20 30]
 [20 30 40]
 [30 40 50]
 [40 50 60]]
Shape: (4, 3)


The dataset has shape `(4, 3)`, meaning it contains four observations
and three numerical features.

Each column will therefore have its own mean.

### Calculating Column Means
To calculate one mean for each feature, I aggregate along `axis=0`.
For a dataset with shape `(4, 3)`, the resulting column-mean array should
have shape `(3,)`.

In [23]:
column_means = data.mean(axis=0)

print("Column means:", column_means)
print("Shape:", column_means.shape)

Column means: [25. 35. 45.]
Shape: (3,)


### Mean-Centering Using Broadcasting
The original dataset has shape `(4, 3)` while the column-means array has
shape `(3,)`.
These shapes are compatible for broadcasting because the three values in
the mean array correspond to the three columns of the dataset.
I will subtract the column means from every row without writing an
explicit Python loop.

In [24]:
centered_data = data - column_means

print("Centered data:")
print(centered_data)

print("Original shape:", data.shape)
print("Mean shape:", column_means.shape)
print("Centered shape:", centered_data.shape)

Centered data:
[[-15. -15. -15.]
 [ -5.  -5.  -5.]
 [  5.   5.   5.]
 [ 15.  15.  15.]]
Original shape: (4, 3)
Mean shape: (3,)
Centered shape: (4, 3)


### Verifying the Centered Data
Mean-centering should produce approximately zero mean for every column.
I will calculate the column means of the centered dataset to verify the
result rather than assuming that the transformation worked correctly.

In [25]:
centered_means = centered_data.mean(axis=0)

print("Means after centering:", centered_means)

Means after centering: [0. 0. 0.]


In [26]:
print("Original column means:", data.mean(axis=0))
print("Centered column means:", centered_data.mean(axis=0))

Original column means: [25. 35. 45.]
Centered column means: [0. 0. 0.]


The original column means were `[25., 35., 45.]`.
After subtracting these means using broadcasting, the mean of each
centered column is approximately zero.
This verifies that the mean-centering operation was applied correctly.
The important point is that a `(3,)` column-mean array was automatically
broadcast across the `(4, 3)` dataset, eliminating the need for an
explicit Python loop.

## Axis-Based Aggregation
The objective of this section is to understand how NumPy aggregation
functions operate along different axes.
I will calculate the mean along `axis=0` and `axis=1`, inspect the
resulting shapes, and explain which dimension is reduced in each case.
Axis-based aggregation is commonly used to summarize numerical datasets.
For example, in a dataset where rows represent observations and columns
represent features, `axis=0` can be used to calculate one statistic per
feature, while `axis=1` can be used to calculate one statistic per
observation.

In [27]:
axis_data = np.array([
    [10, 20, 30, 40],
    [20, 30, 40, 50],
    [30, 40, 50, 60]
])

print(axis_data)
print("Shape:", axis_data.shape)

[[10 20 30 40]
 [20 30 40 50]
 [30 40 50 60]]
Shape: (3, 4)


In [28]:
mean_axis_0 = axis_data.mean(axis=0)

print("Mean along axis=0:", mean_axis_0)
print("Result shape:", mean_axis_0.shape)

Mean along axis=0: [20. 30. 40. 50.]
Result shape: (4,)


In [29]:
mean_axis_1 = axis_data.mean(axis=1)

print("Mean along axis=1:", mean_axis_1)
print("Result shape:", mean_axis_1.shape)

Mean along axis=1: [25. 35. 45.]
Result shape: (3,)


In [30]:
print("Original shape:", axis_data.shape)
print("axis=0 result shape:", mean_axis_0.shape)
print("axis=1 result shape:", mean_axis_1.shape)

Original shape: (3, 4)
axis=0 result shape: (4,)
axis=1 result shape: (3,)


For an array with shape `(3, 4)`:
- `axis=0` aggregates across the 3 rows, leaving 4 column results.
- `axis=1` aggregates across the 4 columns, leaving 3 row results.
A useful verification strategy is to inspect the shape of the result
instead of relying only on memorizing the axis numbers.

In [31]:
print("Sum:", axis_data.sum())
print("Mean:", axis_data.mean())
print("Standard deviation:", axis_data.std())
print("Minimum:", axis_data.min())
print("Maximum:", axis_data.max())

Sum: 420
Mean: 35.0
Standard deviation: 13.844373104863458
Minimum: 10
Maximum: 60


In [32]:
print("Column sums:", axis_data.sum(axis=0))
print("Row sums:", axis_data.sum(axis=1))

Column sums: [ 60  90 120 150]
Row sums: [100 140 180]


## Reshape and View vs Copy
The objective of this section is to understand how NumPy changes the
shape of an array and to investigate whether a reshaped array shares
data with the original array.
I will:
1. Create a one-dimensional array.
2. Reshape it into a two-dimensional array.
3. Verify that the total number of elements remains unchanged.
4. Modify an element through the reshaped array.
5. Check whether the original array is also affected.
Understanding views and copies is important when working with numerical
data because modifying one array may unintentionally affect another array
if both arrays share the same underlying data.

In [33]:
original = np.arange(1, 13)

print("Original array:", original)
print("Shape:", original.shape)
print("Size:", original.size)

Original array: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Shape: (12,)
Size: 12


In [34]:
reshaped = original.reshape(3, 4)

print("Reshaped array:")
print(reshaped)

print("Original shape:", original.shape)
print("Reshaped shape:", reshaped.shape)
print("Original size:", original.size)
print("Reshaped size:", reshaped.size)

Reshaped array:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Original shape: (12,)
Reshaped shape: (3, 4)
Original size: 12
Reshaped size: 12


In [35]:
print("Element count preserved:", original.size == reshaped.size)

Element count preserved: True


In [36]:
print("Before modification:")
print("Original:", original)
print("Reshaped:")
print(reshaped)

Before modification:
Original: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Reshaped:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


In [37]:
reshaped[0, 0] = 999

print("After modification:")
print("Reshaped:", reshaped)
print("Original:", original)

After modification:
Reshaped: [[999   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
Original: [999   2   3   4   5   6   7   8   9  10  11  12]


In [38]:
shares_memory = np.shares_memory(original, reshaped)

print("Do arrays share memory?", shares_memory)

Do arrays share memory? True


### Comparing a View with a Copy
To understand the difference, I will create an independent copy of the
original array.
A copy owns separate data, so modifying the copy should not modify the
original array.

In [39]:
original_copy = original.copy()

original_copy[0] = 111

print("Modified copy:", original_copy)
print("Original array:", original)
print("Do they share memory?", np.shares_memory(original, original_copy))

Modified copy: [111   2   3   4   5   6   7   8   9  10  11  12]
Original array: [999   2   3   4   5   6   7   8   9  10  11  12]
Do they share memory? False


The reshaped array shared memory with the original array, so modifying
the reshaped array also modified the original array.
In contrast, `original.copy()` created an independent array. Modifying
the copy did not affect the original array, and `np.shares_memory()`
confirmed that the two arrays do not share memory.
This demonstrates why the distinction between a view and a copy matters
when modifying NumPy data.

## Jupyter/IPython Exploration Tools
Jupyter provides several IPython tools that make interactive Python
development, exploration, documentation lookup, and performance
measurement easier.
In this section, I will demonstrate:

- `?` for concise object information
- `??` for more detailed source information when available
- `help()` for Python documentation
- `%timeit` for performance benchmarking
- `%run` for executing another Python script
Interactive exploration is useful during AI/ML development because it
allows functions, objects, and code behavior to be investigated without
leaving the development environment.

In [40]:
np.mean?

Signature:      
np.mean(
    a,
    axis=None,
    dtype=None,
    out=None,
    keepdims=<no value>,
    *,
    where=<no value>,
)
Call signature:  np.mean(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function mean at 0x75a3ac78c790>
File:            ~/Documents/week5/.venv/lib/python3.10/site-packages/numpy/_core/fromnumeric.py
Docstring:      
Compute the arithmetic mean along the specified axis.

Returns the average of the array elements.  The average is taken over
the flattened array by default, otherwise over the specified axis.
`float64` intermediate and return values are used for integer inputs.

Parameters
----------
a : array_like
    Array containing numbers whose mean is desired. If `a` is not an
    array, a conversion is attempted.
axis : None or int or tuple of ints, optional
    Axis or axes along which the means are computed. The default is to
    compute the mean of the flattened array.

    If this is a tuple of ints, a mean is perfo

In [41]:
np.mean??

Signature:      
np.mean(
    a,
    axis=None,
    dtype=None,
    out=None,
    keepdims=<no value>,
    *,
    where=<no value>,
)
Call signature:  np.mean(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function mean at 0x75a3ac78c790>
File:            ~/Documents/week5/.venv/lib/python3.10/site-packages/numpy/_core/fromnumeric.py
Source:         
@array_function_dispatch(_mean_dispatcher)
def mean(a, axis=None, dtype=None, out=None, keepdims=np._NoValue, *,
         where=np._NoValue):
    """
    Compute the arithmetic mean along the specified axis.

    Returns the average of the array elements.  The average is taken over
    the flattened array by default, otherwise over the specified axis.
    `float64` intermediate and return values are used for integer inputs.

    Parameters
    ----------
    a : array_like
        Array containing numbers whose mean is desired. If `a` is not an
        array, a conversion is attempted.
    axis : None or int o

In [42]:
help(np.reshape)

Help on _ArrayFunctionDispatcher in module numpy:

reshape(a, /, shape=None, order='C', *, newshape=None, copy=None)
    Gives a new shape to an array without changing its data.
    
    Parameters
    ----------
    a : array_like
        Array to be reshaped.
    shape : int or tuple of ints
        The new shape should be compatible with the original shape. If
        an integer, then the result will be a 1-D array of that length.
        One shape dimension can be -1. In this case, the value is
        inferred from the length of the array and remaining dimensions.
    order : {'C', 'F', 'A'}, optional
        Read the elements of ``a`` using this index order, and place the
        elements into the reshaped array using this index order. 'C'
        means to read / write the elements using C-like index order,
        with the last axis index changing fastest, back to the first
        axis index changing slowest. 'F' means to read / write the
        elements using Fortran-like ind

In [43]:
%timeit np.sum(numbers_array)

424 μs ± 32.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [44]:
# Boolean masking with values greater than 0.5
mask_data = np.array([
    [0.2, 0.7, 0.4],
    [0.9, 0.3, 0.8]
])

mask = mask_data > 0.5

print("Boolean mask:")
print(mask)

print("Values greater than 0.5:")
print(mask_data[mask])

Boolean mask:
[[False  True False]
 [ True False  True]]
Values greater than 0.5:
[0.7 0.9 0.8]
